# 3. Wiring it together

Notebooks 1 and 2 built the pieces. This one is about the glue: what order things run in, who decides to send, what happens when a check fails, and how any of it gets observed.

Still no API key or database -- everything is faked in memory so the control flow is visible.


## Start with the whole thing wrong

Here's roughly what the first version looked like. One function, one
agent, agent sends for itself.


In [1]:
MESSAGES = []          # stands in for Twilio + the messages table

def send_sms(to, body):
    MESSAGES.append((to, body))
    return 'delivered'

def handle_v0(prospect, message):
    reply = f'Thanks for asking about {message}!'   # pretend agent
    send_sms(prospect['phone'], reply)

handle_v0({'phone': '+15551234567'}, 'pricing')
print(MESSAGES)


[('+15551234567', 'Thanks for asking about pricing!')]


Everything wrong with this is invisible until it isn't. No compliance check, no scope check, no grounding, no record of why anything happened.

Build it up properly.


## Step 0: compliance goes first

Before anything else: before triage, before scope, before any model sees the text, the STOP check.

This is first because under CASL an opt-out has to be honoured, and I don't want that guarantee to depend on a model classifying correctly. If someone texts STOP, no LLM should be involved in what happens next.


In [2]:
STOP_KEYWORDS = {'stop', 'unsubscribe', 'quit', 'end', 'cancel', 'stopall', 'arret'}
SUPPRESSIONS = set()     # stands in for the suppressions table

def is_opt_out(body):
    return body.strip().lower() in STOP_KEYWORDS

def handle_opt_out(prospect, body):
    SUPPRESSIONS.add(prospect['phone'])
    return 'opted_out'

print(is_opt_out('STOP'), is_opt_out('stop please'), is_opt_out('how much'))


True False False


Note `'stop please'` returns False — exact match only. That's a real gap the triage agent is meant to cover as a second layer, since it classifies
intent rather than matching strings. Two independent chances to catch it.

The suppression list is deliberately separate from the prospect record. Here's why that matters:


In [3]:
PROSPECTS = {'p1': {'id': 'p1', 'name': 'Wexford Plumbing', 'phone': '+15551234567',
                    'opted_out': False, 'autopilot': False}}

def delete_prospect_naive(pid):
    PROSPECTS.pop(pid, None)

# They opt out, then later I delete them, then I re-import the same lead CSV.
handle_opt_out(PROSPECTS['p1'], 'STOP')
PROSPECTS['p1']['opted_out'] = True
delete_prospect_naive('p1')

PROSPECTS['p1'] = {'id': 'p1', 'name': 'Wexford Plumbing', 'phone': '+15551234567',
                   'opted_out': False, 'autopilot': False}   # re-imported, flag reset

print('prospect says opted_out:', PROSPECTS['p1']['opted_out'])
print('suppression list says  :', PROSPECTS['p1']['phone'] in SUPPRESSIONS)


prospect says opted_out: False
suppression list says  : True


The prospect row forgot. The suppression list didn't.

If the opt-out only lived on the prospect record, that re-import would
have me texting someone who told me to stop. So `suppressions` is
append-only, keyed on the phone number, and has no foreign key to
prospects — the obligation has to outlive the row that happened to carry
it.

And the check belongs inside the send function, not at each call site:


In [4]:
class SendFailed(Exception):
    pass

def send_sms(to, body):
    if to in SUPPRESSIONS:
        raise SendFailed(f'{to} is on the suppression list')
    MESSAGES.append((to, body))
    return 'delivered'

try:
    send_sms('+15551234567', 'hello again!')
except SendFailed as e:
    print('blocked:', e)


blocked: +15551234567 is on the suppression list


One place a message can leave the system, and it's guarded. Add a new code path tomorrow and it inherits the check for free.


## Step 1: two paths, different trust

Not every conversation should be automated. So there's a per-contact flag:

- **review path** (default) — agent drafts, I approve, then it sends
- **autopilot path** (opt-in) — agent drafts, guardrails check, code sends

The gate is one line, and forgetting it in one place is a real bug I shipped: the webhook ran autopilot for *everyone* regardless of the flag. Invisible on localhost because Twilio can't reach it. Would have auto-replied to every prospect the moment I deployed.


In [5]:
REVIEW_QUEUE = {}

def route(prospect, message):
    if is_opt_out(message):
        return handle_opt_out(prospect, message)
    if prospect.get('autopilot'):
        return 'autopilot'
    return 'review'

p = PROSPECTS['p1']
print(route(p, 'how much does it cost'))
p['autopilot'] = True
print(route(p, 'how much does it cost'))
print(route(p, 'STOP'))


review
autopilot
opted_out


## Step 2: the autopilot sequence

Order matters, and each position has a reason.

1. **Scope guardrail** — cheapest, hardest guarantee, so it goes first.
   No point paying for retrieval and a model call on a message that's
   getting blocked anyway.
2. **Retrieval** — empty result is meaningful, not a failure.
3. **Agent drafts** — no tools, returns text.
4. **Grounding guardrail** — checks the draft against what was retrieved.
5. **Send** — plain function call, only if everything above passed.


In [6]:
from dataclasses import dataclass, field

@dataclass
class Outcome:
    action: str
    detail: str = ''
    draft: str = ''

# Fakes from notebook 2, condensed.
def check_scope(m):
    low = m.lower()
    if 'cost' in low or 'price' in low or 'damage' in low:
        return {'allowed': False, 'reason': 'pricing'}
    if 'talk to someone' in low or 'real person' in low:
        return {'allowed': False, 'reason': 'asked for a person'}
    return {'allowed': True, 'reason': ''}

def retrieve(m):
    return [{'id': 'is_it_a_robot', 'answer': "Yes, it's an AI voice agent."}] \
        if 'robot' in m.lower() else []

def agent(m, entries):
    if not entries:
        return {'message': '', 'escalate': True, 'handoff': False}
    return {'message': entries[0]['answer'], 'escalate': False, 'handoff': False}

def check_grounding(draft, entries):
    return {'grounded': bool(entries) or not draft.strip()}

HOLDING = {'pricing': "Good question - let me get you exact numbers. Someone will follow up.",
           'asked for a person': "Of course - someone will reach out shortly.",
           'kb_gap': "Good question - let me check and come back to you shortly."}
print('helpers defined')


helpers defined


Now the sequence itself. The interesting design question is what each failure does -- and specifically, whether it turns autopilot off.


In [7]:
def escalate(prospect, reason, draft='', disable_autopilot=False):
    REVIEW_QUEUE[prospect['id']] = {'reason': reason, 'draft': draft}
    if disable_autopilot:
        prospect['autopilot'] = False

def run_autopilot(prospect, message):
    scope = check_scope(message)
    if not scope['allowed']:
        # A request for a person means a human is now the counterparty.
        # A restricted topic is ONE message the agent may not answer.
        is_handoff = scope['reason'] == 'asked for a person'
        body = HOLDING.get(scope['reason'], '')
        if body:
            send_sms(prospect['phone'], body)
        escalate(prospect, f"scope: {scope['reason']}", disable_autopilot=is_handoff)
        return Outcome('blocked_scope', scope['reason'])

    entries = retrieve(message)
    reply = agent(message, entries)

    if reply['escalate'] or reply['handoff']:
        send_sms(prospect['phone'], HOLDING['kb_gap'])
        escalate(prospect, 'not covered by KB', disable_autopilot=reply['handoff'])
        return Outcome('escalated', 'not covered by KB')

    verdict = check_grounding(reply['message'], entries)
    if not verdict['grounded']:
        # Deliberately NO holding reply here: the agent already spoke and
        # got caught. A second automated message compounds it.
        escalate(prospect, 'unsupported claim', draft=reply['message'])
        return Outcome('blocked_grounding', 'unsupported claim', reply['message'])

    send_sms(prospect['phone'], reply['message'])
    return Outcome('sent', draft=reply['message'])

print('defined')


defined


In [8]:
SUPPRESSIONS.clear(); MESSAGES.clear(); REVIEW_QUEUE.clear()
p = {'id': 'p1', 'name': 'Wexford', 'phone': '+15551234567', 'autopilot': True}

for m in ['is this a robot', 'how much does it cost', 'what is the weather',
          'can I talk to someone']:
    before = p['autopilot']
    o = run_autopilot(p, m)
    print(f"{m!r:26} {o.action:18} autopilot {before} -> {p['autopilot']}")


'is this a robot'          sent               autopilot True -> True
'how much does it cost'    blocked_scope      autopilot True -> True
'what is the weather'      escalated          autopilot True -> True
'can I talk to someone'    blocked_scope      autopilot True -> False


That last column is the interesting one, and I have now got it wrong in both directions.

**Version 1.** Every refusal called `set_autopilot(False)`. So the first pricing question turned autopilot off permanently and every message after
that dropped into the review queue. From the outside it looked exactly like autopilot being broken - I spent a while hunting a bug that was my
own deliberate behaviour, unsurfaced.

**Version 2.** Narrowed it: only a genuine handoff stops autopilot. A prospect asking one pricing question hasn't stopped being an autopilot
conversation.

**Version 3.** A live transcript settled it:

```
"What's the damage"            -> "someone will text you shortly"
"Is this gonna break the bank" -> (blocked, no reply)
"?"                            -> "let me check and come back to you"
"How much is it"               -> "someone will text you shortly"
```

Four pricing attempts, three promises of follow-up, nobody followed up. A bot cheerfully deflecting the same question repeatedly is worse than a
queue entry. So scope blocks disable autopilot again, and the operator re-enables from the console when they have handled it - which also clears
`needs_human`.

The code below shows version 2. I have left it that way on purpose: the argument for it is genuinely reasonable, and it took real traffic rather
than reasoning to show it was wrong.


In [10]:
for to, body in MESSAGES:
    print(f'{to}  {body[:70]}')

+15551234567  Yes, it's an AI voice agent.
+15551234567  Good question - let me get you exact numbers. Someone will follow up.
+15551234567  Good question - let me check and come back to you shortly.
+15551234567  Of course - someone will reach out shortly.


Two of those four runs produced no outbound message at all, and one
produced a holding reply that says nothing about *why*.

From the message log, 'the grounding guardrail caught an invented claim'
and 'nothing ran' are indistinguishable. Those are exactly the runs I
most want to inspect — they're where the guardrails either earned their
keep or over-blocked something answerable.

So: record the steps.


In [11]:
import time

TRACES = []

class Trace:
    def __init__(self, prospect_id, trigger):
        self.prospect_id, self.trigger = prospect_id, trigger
        self.steps, self.outcome = [], 'incomplete'
        self._t0 = time.perf_counter()

    def step(self, name, status='ok', **detail):
        self.steps.append({'name': name, 'status': status,
                           'at_ms': round((time.perf_counter() - self._t0) * 1000),
                           **detail})

    def finish(self, outcome):
        self.outcome = outcome
        try:
            TRACES.append(self)
        except Exception:
            pass          # observability must never break what it observes

print('defined')


defined


Threading it through is mechanical — a `tr.step(...)` after each decision.
The one design choice: `finish()` swallows its own errors. A tracing
failure taking down the send path it's watching would be a bad trade.


In [12]:
def run_autopilot_traced(prospect, message):
    tr = Trace(prospect['id'], message)

    scope = check_scope(message)
    tr.step('scope_guardrail', 'ok' if scope['allowed'] else 'blocked', reason=scope['reason'])
    if not scope['allowed']:
        is_handoff = scope['reason'] == 'asked for a person'
        body = HOLDING.get(scope['reason'], '')
        if body:
            send_sms(prospect['phone'], body)
            tr.step('holding_reply', 'ok', topic=scope['reason'])
        escalate(prospect, f"scope: {scope['reason']}", disable_autopilot=is_handoff)
        tr.step('autopilot_state', 'ok',
                state='disabled - handoff' if is_handoff else 'left on')
        tr.finish('blocked_scope')
        return Outcome('blocked_scope', scope['reason'])

    entries = retrieve(message)
    tr.step('kb_retrieval', 'ok' if entries else 'skipped',
            matched=[e['id'] for e in entries])

    reply = agent(message, entries)
    tr.step('sdr_agent', 'ok', escalate=reply['escalate'], draft=reply['message'][:60])

    if reply['escalate']:
        send_sms(prospect['phone'], HOLDING['kb_gap'])
        tr.step('holding_reply', 'ok', topic='kb_gap')
        escalate(prospect, 'not covered by KB')
        tr.finish('escalated')
        return Outcome('escalated')

    verdict = check_grounding(reply['message'], entries)
    tr.step('grounding_guardrail', 'ok' if verdict['grounded'] else 'blocked')
    if not verdict['grounded']:
        escalate(prospect, 'unsupported claim', draft=reply['message'])
        tr.finish('blocked_grounding')
        return Outcome('blocked_grounding')

    send_sms(prospect['phone'], reply['message'])
    tr.step('send_sms', 'ok', to=prospect['phone'])
    tr.finish('sent')
    return Outcome('sent', draft=reply['message'])

MESSAGES.clear(); TRACES.clear(); REVIEW_QUEUE.clear()
p = {'id': 'p1', 'name': 'Wexford', 'phone': '+15551234567', 'autopilot': True}
for m in ['is this a robot', 'how much does it cost', 'what is the weather']:
    run_autopilot_traced(p, m)
    p['autopilot'] = True     # reset so all three actually run

for t in TRACES:
    print(f'\n{t.outcome}  <- {t.trigger!r}')
    for s in t.steps:
        mark = {'ok': 'o', 'blocked': 'X', 'skipped': '.'}.get(s['status'], '?')
        extra = ' '.join(f'{k}={v}' for k, v in s.items()
                         if k not in ('name', 'status', 'at_ms'))
        print(f"  {mark} {s['name']:22} {extra}")



sent  <- 'is this a robot'
  o scope_guardrail        reason=
  o kb_retrieval           matched=['is_it_a_robot']
  o sdr_agent              escalate=False draft=Yes, it's an AI voice agent.
  o grounding_guardrail    
  o send_sms               to=+15551234567

blocked_scope  <- 'how much does it cost'
  X scope_guardrail        reason=pricing
  o holding_reply          topic=pricing
  o autopilot_state        state=left on

escalated  <- 'what is the weather'
  o scope_guardrail        reason=
  . kb_retrieval           matched=[]
  o sdr_agent              escalate=True draft=
  o holding_reply          topic=kb_gap


Now a blocked run is legible. `X scope_guardrail reason=pricing` followed by `o holding_reply` tells me exactly what happened and what the prospect
received, on a run that left no message in the conversation.

## Step 4: one sequence, two callers

Messages arrive two ways: a Twilio webhook when deployed, and polling the Twilio API when running locally (Twilio can't reach localhost).

Both need this identical sequence. The obvious move is to write it in both places, and that's how the autopilot-flag bug happened — the poller
checked it, the webhook didn't.

So the sequence lives in one module and both callers import it.


In [13]:
def inbound(prospect, message, source):
    """What both the webhook and the poller reduce to."""
    if is_opt_out(message):
        return handle_opt_out(prospect, message)
    if prospect.get('autopilot'):
        return run_autopilot_traced(prospect, message).action
    REVIEW_QUEUE[prospect['id']] = {'reason': message, 'draft': '(drafted for review)'}
    return 'queued_for_review'

p = {'id': 'p1', 'phone': '+15551234567', 'autopilot': False}
print('autopilot off:', inbound(p, 'is this a robot', 'webhook'))
p['autopilot'] = True
print('autopilot on :', inbound(p, 'is this a robot', 'poller'))
print('stop         :', inbound(p, 'STOP', 'poller'))


autopilot off: queued_for_review
autopilot on : sent
stop         : opted_out


## The whole map

| built here | in the repo |
|---|---|
| `is_opt_out`, `STOP_KEYWORDS` | inline in `webhook.py` / `poll_inbound.py`, before triage |
| `SUPPRESSIONS` | `suppressions` table, migration 007 |
| `send_sms` with the guard | `app/tools/twilio_sms.py` |
| `check_scope` | `app/agents/guardrails.py` |
| `retrieve` | `app/kb/loader.py` (notebook 1) |
| `agent` | `app/agents/sdr_agent.py`, no tools, structured output |
| `check_grounding` | `app/agents/guardrails.py` |
| `run_autopilot_traced` | `app/agents/autopilot.py` |
| `Trace` | `app/observability.py` + `agent_traces` table, migration 008 |
| `inbound` | the shared call in `webhook.py` and `poll_inbound.py` |

## What I'd carry to the next agent I build

**Cheapest and most certain check first.** Keywords before classifier, classifier before retrieval, retrieval before the model. Each layer only pays for what got past the one before it.

**Agents return decisions; code performs them.** Anything irreversible -- sending, disabling a flag, charging a card — belongs after the checks, in code. An output guardrail that runs after a tool call is watching something that already happened.

**A refusal is not a failure state.** Every blocked path here still does something useful: acknowledges the prospect, queues for a human, records why. Silence is the actual failure.

**Record the runs that produce nothing.** Those are the ones worth reading, and they're invisible by default.

**One copy of a compliance-relevant sequence.** Two copies drift, and the drift is silent until it's a legal problem.
